# Pick stars for saturation fitting!

## Do a big query, then use this to test the i/o module

In [1]:
import os
import importlib
import pprint

import matplotlib.pyplot as plt
#%matplotlib widget
%matplotlib inline

import math
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic, binned_statistic_2d
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack, hstack

In [2]:
from astropy.stats import sigma_clipped_stats

In [3]:
from astropy.visualization import LinearStretch, LuptonAsinhZscaleStretch, SinhStretch
from astropy.visualization import imshow_norm, simple_norm
from astropy.visualization import MinMaxInterval, ZScaleInterval, PercentileInterval
from astropy.visualization import SqrtStretch, ImageNormalize

In [4]:
import lsst.afw.image as afwImage
import lsst.afw.display as afwDisplay
import lsst.geom
import lsst.afw.geom as afwGeom

import lsst.daf.butler as dafButler
import lsst.pipe.base

In [5]:
from lsst.analysis.ap import apdb
from lsst.analysis.ap import nb_utils
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles,
)
from lsst.utils.plotting import stars_cmap
from lsst.utils.plotting import publication_plots
publication_plots.set_rubin_plotstyle()

clrs = get_multiband_plot_colors()
bands_dict = publication_plots.get_band_dicts()

        Use pytest instead. [astropy.tests.runner]


        Use pytest instead. [astropy.utils.decorators]


In [6]:
from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib.ticker import PercentFormatter
import matplotlib.gridspec as gridspec

In [7]:
import matplotlib.patheffects as pathEffects

from lsst.utils.plotting import (                                                                             
         galaxies_cmap,                                                                                        
         galaxies_color,                                                                                       
         make_figure,                                                                                          
         stars_cmap,                                                                                           
         stars_color,                                                                                          
         set_rubin_plotstyle,                                                                                  
         divergent_cmap,                                                                                       
         accent_color,)    
set_rubin_plotstyle()

In [8]:
# All figures in this directory
from pathlib import Path

In [9]:
afwDisplay.setDefaultBackend("firefly")
afwDisplay.setDefaultBackend("matplotlib")


In [10]:
butler = dafButler.Butler('dp1', collections='LSSTComCam/DP1')

In [11]:
print(butler)

Butler(collections=DirectButlerCollections(defaults=('LSSTComCam/DP1',)), run=None, datastore='file:///sdf/group/rubin/repo/dp1/', registry='PostgreSQL@lsstdb1:dp1')


## Do the source query first. 

* Pick stars, with many flux estimators.
* Only r or g filter
* Near the center of ECDFS
* With a limit on flux so they are brighter than mag 20

In [58]:
from lsst.rsp import get_tap_service
service = get_tap_service("tap")
assert service is not None

lsst_mag_limit = 20.

query = (
    "SELECT sourceId, visit, ra, dec, band, detector, blendedness_abs, "
    "psfFlux, psfFluxErr, psfFlux_flag, psfFlux_apCorr, psfFlux_apCorrErr, psfFlux_area,"
    "ap03Flux, ap03FluxErr, ap03Flux_flag, calibFlux, calibFluxErr, "
    "ap09Flux, ap09FluxErr, ap09Flux_flag, "
    "ap12Flux, ap12FluxErr, ap12Flux_flag, "
    "ap17Flux, ap17FluxErr, ap17Flux_flag, "
    "calib_astrometry_used, calib_photometry_used, calib_psf_used, "
    "pixelFlags_bad, pixelFlags_saturated, pixelFlags_saturatedCenter, localBackground_flag "
    "FROM dp1.Source " 
    "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), CIRCLE('ICRS', 53, -28, 1.75))=1  " 
    f"AND psfFLux > {10 **(-0.4 * (lsst_mag_limit - 31.4))} " 
    "AND (band = 'r' OR band = 'g') "
    "ORDER BY psfFlux ASC "
        )

job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
results = job.fetch_result().to_table()

In [14]:
results

sourceId,visit,ra,dec,band,detector,blendedness_abs,psfFlux,psfFluxErr,psfFlux_flag,psfFlux_apCorr,psfFlux_apCorrErr,psfFlux_area,ap03Flux,ap03FluxErr,ap03Flux_flag,calibFlux,calibFluxErr,ap09Flux,ap09FluxErr,ap09Flux_flag,ap12Flux,ap12FluxErr,ap12Flux_flag,ap17Flux,ap17FluxErr,ap17Flux_flag,calib_astrometry_used,calib_photometry_used,calib_psf_used,pixelFlags_bad,pixelFlags_saturated,pixelFlags_saturatedCenter,localBackground_flag
,,deg,deg,,,,nJy,nJy,,,,pix,nJy,nJy,,nJy,nJy,nJy,nJy,,nJy,nJy,,nJy,nJy,,,,,,,,
int64,int64,float64,float64,str1,int32,float64,float64,float64,bool,float64,float64,float32,float64,float64,bool,float64,float64,float64,float64,bool,float64,float64,bool,float64,float64,bool,bool,bool,bool,bool,bool,bool,bool
600460946475843611,2024121000405,53.31827197153322,-28.299317900792072,r,6,0.0,36308.20531221978,2829.979609766467,False,0.9858255164207499,0.0,98.7073,17294.284537101797,1507.540621038883,False,24916.850340163488,7663.963320510074,39631.70791666852,4517.4385597724695,False,36275.30071127381,6031.735874498474,False,48150.99769008265,8642.821591479833,False,False,False,False,False,False,False,False
600460946472698002,2024121000405,52.901807444841054,-28.407004203787608,r,0,0.0,36308.225110987834,3099.704036211506,False,0.9856904447912442,0.0,105.878,16458.45845045563,1592.822210996669,False,50773.72195111007,8132.634684628732,42893.67578846947,4794.10119911122,False,48399.30156607581,6404.97697457561,False,48569.67831262031,9157.570262595951,False,False,False,False,False,False,False,False
600438916245882889,2024120500107,53.2435978945759,-27.83199749308097,r,5,0.0,36308.2320062036,233.74736966893593,True,0.991300409204809,0.0,82.171,18754.916290297202,135.11134952894878,True,--,--,--,--,True,--,--,True,--,--,True,False,False,False,True,False,False,True
600408129787659360,2024112800106,53.08927418445556,-28.18881607334168,r,8,0.0,36308.27849135756,230.26819272029027,False,1.004240442492121,0.0,75.4995,18591.416858612098,137.16068702660968,False,59006.08959283898,520.6082172610181,56472.26525555988,336.96542891810674,False,66006.52658243844,428.22183972862723,False,76247.05126251756,583.7284224861155,False,False,False,False,False,False,False,False
600412544877724581,2024112900233,53.10555556084018,-28.123554121779744,g,4,0.0,36308.576926936716,204.19629295522503,False,0.9915431865267988,0.0,71.6712,20457.230057312863,124.60503949936066,False,36964.21025898603,384.12378165400503,36035.77789696281,253.49038956617827,False,37290.029098368694,316.32084609144243,False,38024.58986060901,428.37950290424317,False,False,False,True,False,False,False,False
600403739659993416,2024112700165,53.21680385375117,-28.332459706969633,g,8,0.0,36308.660183191685,230.60714973611564,False,1.0047240063645244,0.0,125.25,14376.32602207749,111.38244594524798,False,35676.38801982487,389.11948069051095,33504.32225145206,252.13057252184814,False,35476.51863491668,316.80022991539454,False,36499.43403605384,430.5462957058939,False,False,False,True,False,False,False,False
600456536753439583,2024120900318,53.3807716416791,-27.9429184058744,g,8,0.0,36308.7216826187,289.8502394723407,False,0.9752994132777087,0.0,171.654,11805.039794007504,123.68104873432927,False,35672.06295771322,505.2660247495935,33189.70039984245,312.76509429688826,False,36220.00284542966,400.7356839441281,False,38654.1333479882,553.5252193066626,False,False,False,True,False,False,False,False
600456536615551567,2024120900317,52.88555185252231,-28.18179877307477,g,1,0.0,36308.91670991153,276.2557676284172,False,0.9753659249707782,0.0,173.477,9400.39095415824,114.79923946392543,False,48900.447416775045,505.78090991006957,43158.3258179967,314.04631977973764,False,54265.22354219243,403.42561929008247,False,64306.68171343538,551.813703243213,False,False,False,False,False,False,False,False


In [15]:
from sklearn.linear_model import LinearRegression

In [16]:
from scipy import linalg

In [17]:
from scipy.stats import linregress

## Use now the Monster

* Query the monster catalog at the positions of stars
* Use the HTM7 trixels to pick up the calibration stars in ECDFS

In [18]:
butler.registry.queryDatasetTypes('*monster*')

[DatasetType('the_monster_20250219', {htm7}, SimpleCatalog)]

In [19]:
monsterquery = butler.registry.queryDatasets('the_monster_20250219')
monsterquery.count()
# list(butler.registry.queryDatasets('the_monster_20250219'))

131072

In [20]:
butler.query_datasets('skyMap')

[DatasetRef(DatasetType('skyMap', {skymap}, SkyMap), {skymap: 'lsst_cells_v1'}, run='skymaps', id=64c16b0a-0a1a-4f78-81e5-f49a9328c388)]

In [21]:
skymap = butler.get('skyMap')

In [22]:
import esutil

In [23]:
htm = esutil.htm.HTM(7)
htm_indices = set(htm.lookup_id(results["ra"], results["dec"]))  # type: ignore

In [24]:
htm_indices

{np.int64(147048),
 np.int64(147049),
 np.int64(147051),
 np.int64(147172),
 np.int64(147173),
 np.int64(147174),
 np.int64(147175)}

In [25]:
monster_cats = [butler.get('the_monster_20250219', htm7=htm_indx).asAstropy() for htm_indx in htm_indices]

In [26]:
monsterCat = vstack(monster_cats)

In [27]:
len(monsterCat)

12020

In [28]:
monsterCat['coord_ra_deg'] = np.rad2deg(monsterCat['coord_ra'])
monsterCat['coord_dec_deg'] = np.rad2deg(monsterCat['coord_dec'])

In [29]:
stardata = results

In [30]:
stars = SkyCoord(ra=stardata['ra'], dec=stardata['dec'])

Now make a crossmatch between stars and monster catalog

This would clean galaxies, and use stars we have information from
other external surveys, plus transformations to LSST bandpasses

In [31]:
# Convert to SkyCoord
stars = SkyCoord(ra=stardata['ra'], dec=stardata['dec'])
monsters = SkyCoord(ra=monsterCat['coord_ra_deg'].value*u.deg, dec=monsterCat['coord_dec_deg'].value*u.deg)

# Crossmatch (one-to-one, find nearest neighbor)
idx, d2d, d3d = stars.match_to_catalog_sky(monsters)

# Example: Add matched index and separation to stardata
stardata['monsterCat_index'] = idx
stardata['match_sep_arcsec'] = d2d.arcsec
stardata['match_monsterCat'] = d2d.arcsec < 0.75

# Merge matched rows for more information (optional)
matched_monsters = monsterCat[idx]
crossmatched = hstack([stardata, matched_monsters])


In [32]:
selection = crossmatched[crossmatched['match_monsterCat'] == True]

In [33]:
selection

sourceId,visit,ra,dec,band,detector,blendedness_abs,psfFlux,psfFluxErr,psfFlux_flag,psfFlux_apCorr,psfFlux_apCorrErr,psfFlux_area,ap03Flux,ap03FluxErr,ap03Flux_flag,calibFlux,calibFluxErr,ap09Flux,ap09FluxErr,ap09Flux_flag,ap12Flux,ap12FluxErr,ap12Flux_flag,ap17Flux,ap17FluxErr,ap17Flux_flag,calib_astrometry_used,calib_photometry_used,calib_psf_used,pixelFlags_bad,pixelFlags_saturated,pixelFlags_saturatedCenter,localBackground_flag,monsterCat_index,match_sep_arcsec,match_monsterCat,id,coord_ra,coord_dec,phot_g_mean_flux,phot_bp_mean_flux,phot_rp_mean_flux,phot_g_mean_fluxErr,phot_bp_mean_fluxErr,phot_rp_mean_fluxErr,coord_raErr,coord_decErr,epoch,pm_ra,pm_dec,pm_raErr,pm_decErr,pm_flag,parallax,parallaxErr,parallax_flag,coord_ra_coord_dec_Cov,coord_ra_pm_ra_Cov,coord_ra_pm_dec_Cov,coord_ra_parallax_Cov,coord_dec_pm_ra_Cov,coord_dec_pm_dec_Cov,coord_dec_parallax_Cov,pm_ra_pm_dec_Cov,pm_ra_parallax_Cov,pm_dec_parallax_Cov,astrometric_excess_noise,monster_ComCam_g_flux,monster_ComCam_g_fluxErr,monster_ComCam_g_source_flag,monster_ComCam_i_flux,monster_ComCam_i_fluxErr,monster_ComCam_i_source_flag,monster_ComCam_r_flux,monster_ComCam_r_fluxErr,monster_ComCam_r_source_flag,monster_ComCam_y_flux,monster_ComCam_y_fluxErr,monster_ComCam_y_source_flag,monster_ComCam_z_flux,monster_ComCam_z_fluxErr,monster_ComCam_z_source_flag,monster_DES_g_flux,monster_DES_g_fluxErr,monster_DES_g_source_flag,monster_DES_i_flux,monster_DES_i_fluxErr,monster_DES_i_source_flag,monster_DES_r_flux,monster_DES_r_fluxErr,monster_DES_r_source_flag,monster_DES_y_flux,monster_DES_y_fluxErr,monster_DES_y_source_flag,monster_DES_z_flux,monster_DES_z_fluxErr,monster_DES_z_source_flag,monster_LATISS_g_flux,monster_LATISS_g_fluxErr,monster_LATISS_g_source_flag,monster_LATISS_i_flux,monster_LATISS_i_fluxErr,monster_LATISS_i_source_flag,monster_LATISS_r_flux,monster_LATISS_r_fluxErr,monster_LATISS_r_source_flag,monster_LATISS_y_flux,monster_LATISS_y_fluxErr,monster_LATISS_y_source_flag,monster_LATISS_z_flux,monster_LATISS_z_fluxErr,monster_LATISS_z_source_flag,monster_ComCam_u_flux,monster_ComCam_u_fluxErr,monster_ComCam_u_source_flag,monster_SDSS_u_flux,monster_SDSS_u_fluxErr,monster_SDSS_u_source_flag,coord_ra_deg,coord_dec_deg
,,deg,deg,,,,nJy,nJy,,,,pix,nJy,nJy,,nJy,nJy,nJy,nJy,,nJy,nJy,,nJy,nJy,,,,,,,,,,,,,rad,rad,nJy,nJy,nJy,nJy,nJy,nJy,rad,rad,d,rad,rad,rad / yr,rad / yr,,rad,rad,,rad2,rad2 / yr,rad2 / yr,rad2,rad2 / yr,rad2 / yr,rad2,rad2 / yr2,rad2 / yr,rad2 / yr,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,nJy,nJy,,rad,rad
int64,int64,float64,float64,str1,int32,float64,float64,float64,bool,float64,float64,float32,float64,float64,bool,float64,float64,float64,float64,bool,float64,float64,bool,float64,float64,bool,bool,bool,bool,bool,bool,bool,bool,int64,float64,bool,int64,float64,float64,float64,float64,float64,float64,float64,float64,float32,float32,float64,float64,float64,float32,float32,uint8,float64,float64,uint8,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64,int32,float64,float64
600460946475843611,2024121000405,53.31827197153322,-28.299317900792072,r,6,0.0,36308.20531221978,2829.979609766467,False,0.9858255164207499,0.0,98.7073,17294.284537101797,1507.540621038883,False,24916.850340163488,7663.963320510074,39631.70791666852,4517.4385597724695,False,36275.30071127381,6031.735874498474,False,48150.99769008265,8642.821591479833,False,False,False,False,False,False,False,False,5786,0.2839292859566078,True,5057422740928144640,0.9305785421593299,-0.49391512112970

In [42]:
selection.write('stars_crossmatched_monster.pqt', format='parquet', overwrite=True)

Now we find the set of visits that we picked stars from.

In [43]:
visitset = "("+f",".join([str(avisit) for avisit in set(selection['visit'])])+")"

In [52]:
queryVisits = (
    "SELECT * "
    "FROM dp1.CcdVisit " 
    "WHERE visitId IN "+visitset 
)

job = service.submit_job(queryVisits)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
ccdvisitstable = job.fetch_result().to_table()

Job phase is COMPLETED


Now we pick CCD visit tables 

In [53]:
ccdvisitstable.write('ccdvisit_table.ecsv', format='ecsv', overwrite=True)

In [54]:
queryVisits = (
    "SELECT * "
    "FROM dp1.Visit " 
    "WHERE visit IN "+visitset 
)

job = service.submit_job(queryVisits)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
visitstable = job.fetch_result().to_table()

Job phase is COMPLETED


In [55]:
visitstable.write('visit_table.ecsv', format='ecsv', overwrite=True)

In [56]:
ccdvisitstable

astromOffsetMean,astromOffsetStd,band,ccdVisitId,darkTime,dec,detector,effTime,effTimePsfSigmaScale,effTimeSkyBgScale,effTimeZeroPointScale,expTime,llcdec,llcra,lrcdec,lrcra,magLim,maxDistToNearestPsf,nPsfStar,obsStart,physical_filter,pixelScale,psfApCorrSigmaScaledDelta,psfApFluxDelta,psfSigma,psfStarDeltaE1Median,psfStarDeltaE1Scatter,psfStarDeltaE2Median,psfStarDeltaE2Scatter,psfStarDeltaSizeMedian,psfStarDeltaSizeScatter,psfStarScaledDeltaSizeScatter,psfTraceRadiusDelta,ra,seeing,skyBg,skyNoise,skyRotation,ulcdec,ulcra,urcdec,urcra,xSize,ySize,zenithDistance,zeroPoint,visitId,expMidptMJD,expMidpt,obsStartMJD
arcsec,arcsec,,,s,deg,,s,,,,s,deg,deg,deg,deg,mag,pix,,,,arcsec / pix,,,pix,,,,,pix,pix,,pix,deg,arcsec,adu,adu,deg,deg,deg,deg,deg,pix,pix,deg,mag,,d,,d
float64,float64,object,int64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,object,object,float32,float64,float64,float32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float32,float32,float64,float64,float64,float64,float64,int64,int64,float32,float32,int64,float64,object,float64
0.0119103,0.00723752,r,1145019889152,30.4398,-28.19118466571906,0,23.3037,0.674719,1.11188,1.03543,30.0,-28.276620734617,52.85263510593473,-28.326588905094983,53.09916752695214,24.5735,1025.56,56,2024-11-09T06:13:11.028,r_03,0.200393,0.0127647,0.100171,1.89917,0.00173806,0.0127653,0.00119829,0.0108095,-0.00322224,0.0152097,0.00802887,0.232188,53.00490912491905,0.8961970807514064,889.033,32.9384,102.79934100531,-28.0557519322531,52.91082291320832,-28.105687923007274,53.15675030958463,4071,3999,16.0632,32.0717,2024110800246,60623.25932903928,2024-11-09T06:13:26.028,60623.259155428175
0.0107645,0.00567092,r,1145019889153,30.4398,-27.962022324033423,1,24.1516,0.710092,1.09233,1.0379,30.0,-28.04724738582031,52.91287292075814,-28.097425479796303,53.15872609276538,24.6319,645.261,79,2024-11-09T06:13:11.028,r_03,0.200324,0.0094962,0.109776,1.85126,0.000119366,0.0118748,0.000550285,0.0126894,-0.000388883,0.0146253,0.00788028,0.0805953,53.06473651356463,0.8732876028034549,904.946,29.5841,102.79934100531,-27.82651722495108,52.970902764003604,-27.876660595073368,53.21625674742541,4071,3999,16.0606,32.073,2024110800246,60623.25932903928,2024-11-09T06:13:26.028,60623.259155428175
0.0101954,0.00545996,r,1145019889154,30.4398,-27.73242310816508,2,23.2671,0.671652,1.11448,1.03611,30.0,-27.817874982321907,52.97297803385288,-27.8677349068775,53.21838585026093,24.6158,784.07,75,2024-11-09T06:13:11.028,r_03,0.20039,0.0113236,0.0463592,1.9035,-0.000696734,0.0121295,0.00192478,0.0113272,-0.00254452,0.0158764,0.00827613,0.128863,53.12430114599026,0.8982269412176027,886.962,31.6826,102.79934100531,-27.59690850342436,53.03031927682086,-27.64676053294207,53.27533373287446,4071,3999,16.0614,32.0721,2024110800246,60623.25932903928,2024-11-09T06:13:26.028,60623.259155428175
0.00763241,0.0041554,r,1145019889155,30.4398,-28.243371667827503,3,23.0161,0.663748,1.11756,1.03428,30.0,-28.32909612815888,53.1132365032838,-28.3786008376788,53.35985956166373,24.6035,970.123,71,2024-11-09T06:13:11.028,r_03,0.200328,0.00938753,0.0973662,1.9148,-0.000814107,0.0108858,0.0018549,0.0124462,-0.00190054,0.0128662,0.0067469,0.157791,53.26513447668298,0.903277871845216,884.517,32.2186,102.79934100531,-28.10816641092561,53.17066143271923,-28.157553668739677,53.41672458090698,4071,3999,15.8281,32.0711,2024110800246,60623.25932903928,2024-11-09T06:13:26.028,60623.259155428175
0.00720748,0.00398673,r,1145019889156,30.4398,-28.014305184493097,4,23.4676,0.683307,1.10266,1.03822,30.0,-28.099942903851062,53.172688157876486,-28.149435244343508,53.418705594976075,24.6334,727.46,79,2024-11-09T06:13:11.028,r_03,0.200265,0.0215212,0.135198,1.88719,0.0015209,0.0138282,-0.00111395,0.0129848,-0.00125035,0.0145477,0.00775373,0.180105,53.324199511753605,0.8899787908721435,896.465,32.1111,102.79934100531,-27.879117410161165,53.22992441475596,-27.92850606730432

In [57]:
visitstable

airmass,altitude,azimuth,band,dec,expTime,obsStart,physical_filter,ra,skyRotation,visit,zenithDistance,expMidptMJD,expMidpt,obsStartMJD
,deg,deg,,deg,s,,,deg,deg,,deg,d,,d
float64,float64,float64,object,float64,float64,object,object,float64,float64,int64,float64,float64,object,float64
1.03814024464709,74.4073000879621,273.947883920456,r,-28.0723466054328,30.0,2024-11-09T06:13:11.028,r_03,53.3274246636802,102.79934100531,2024110800246,15.5926999120379,60623.25932903928,2024-11-09T06:13:26.028,60623.259155428175
1.03979582779195,74.0834594083086,273.463828946346,r,-28.1312112068802,30.0,2024-11-09T06:13:59.959,r_03,53.1413782855617,102.799321817097,2024110800247,15.916540591691401,60623.25989537033,2024-11-09T06:14:14.959,60623.25972175922
1.0431643110249,73.4458487097539,272.690602893552,r,-28.2085120405563,30.0,2024-11-09T06:17:12.837,r_03,53.1891579665548,102.799582415553,2024110800250,16.554151290246097,60623.262127748836,2024-11-09T06:17:27.837,60623.26195413773
1.04506638636346,73.0974021460806,272.822772674086,r,-28.0984834194087,30.0,2024-11-09T06:18:01.783,r_03,53.0158795112189,102.799560434871,2024110800251,16.902597853919403,60623.2626942534,2024-11-09T06:18:16.783,60623.26252064229
1.04922264166105,72.3647772746761,271.928591170615,r,-28.2058717329746,30.0,2024-11-09T06:21:12.168,r_03,52.9367180670944,102.799261214045,2024110800254,17.6352227253239,60623.2648977835,2024-11-09T06:21:27.168,60623.26472417239
1.04974881155767,72.2741667124165,272.274071249141,r,-28.0840881235852,30.0,2024-11-09T06:22:01.240,r_03,53.0660897082471,102.799221456629,2024110800255,17.725833287583498,60623.265465752265,2024-11-09T06:22:16.240,60623.26529214116
1.0544557351707,71.4891972461329,272.047374537714,r,-27.9861418678142,30.0,2024-11-09T06:25:13.724,r_03,52.9838725416809,102.799286374179,2024110800258,18.510802753867097,60623.2676935763,2024-11-09T06:25:28.724,60623.26751996519
1.05548229559587,71.3235387733284,271.572324026355,r,-28.0993826432712,30.0,2024-11-09T06:26:02.870,r_03,52.9697437412314,102.799274656101,2024110800259,18.676461226671606,60623.268262389945,2024-11-09T06:26:17.870,60623.26808877884
